# Shared memory tiling and a tiled matmul

Why a naive matrix multiply is starved for data, how shared-memory tiles cut global memory traffic by the tile size, why that alone still isn't enough, and how register blocking (each thread computing a small grid of outputs) raises arithmetic intensity until the kernel can approach the compute roofline. You write three versions and compare them with cuBLAS.

This notebook holds the code from the lesson **[Shared memory tiling and a tiled matmul](https://inferenceacademy.com/learn/kern-cuda/shared-memory-tiling-and-a-tiled-matmul)** on Inference Academy, in the same order. Read the lesson for the explanations, diagrams and exercises, and run the cells here as you go.

**Before you start:** choose *Runtime → Change runtime type → T4 GPU*, then run the first cell to check the GPU.

In [ ]:
!nvidia-smi

## Version 1: one thread per output

Snippet from the lesson (add it to your program; not run on its own):

```cpp
__global__ void sgemm_naive(const float* A, const float* B, float* C, int M, int N, int K) {
  int r = blockIdx.y * blockDim.y + threadIdx.y;
  int c = blockIdx.x * blockDim.x + threadIdx.x;
  if (r >= M || c >= N) return;
  float acc = 0.f;
  for (int k = 0; k < K; k++) acc += A[r * K + k] * B[k * N + c];
  C[r * N + c] = acc;
}
// launch: dim3 block(32, 32); dim3 grid((N + 31) / 32, (M + 31) / 32);
```

## Version 2: shared-memory tiles

Snippet from the lesson (add it to your program; not run on its own):

```cpp
#define T 32
__global__ void sgemm_tiled(const float* A, const float* B, float* C, int M, int N, int K) {
  __shared__ float As[T][T], Bs[T][T];
  int r = blockIdx.y * T + threadIdx.y, c = blockIdx.x * T + threadIdx.x;
  float acc = 0.f;
  for (int k0 = 0; k0 < K; k0 += T) {
    As[threadIdx.y][threadIdx.x] = (r < M && k0 + threadIdx.x < K) ? A[r * K + k0 + threadIdx.x] : 0.f;
    Bs[threadIdx.y][threadIdx.x] = (k0 + threadIdx.y < K && c < N) ? B[(k0 + threadIdx.y) * N + c] : 0.f;
    __syncthreads();                         // the tile is fully loaded
    for (int k = 0; k < T; k++) acc += As[threadIdx.y][k] * Bs[k][threadIdx.x];
    __syncthreads();                         // everyone is done reading before the next load
  }
  if (r < M && c < N) C[r * N + c] = acc;
}
```

### The new bottleneck: shared memory

## Version 3: register blocking

Snippet from the lesson (add it to your program; not run on its own):

```cpp
#define BM 64
#define BN 64
#define BK 16
#define TM 4
#define TN 4      // 256 threads, arranged 16 x 16, each computes a 4 x 4 patch

__global__ void sgemm_reg(const float* A, const float* B, float* C, int M, int N, int K) {
  __shared__ float As[BM][BK], Bs[BK][BN];
  int tx = threadIdx.x % 16, ty = threadIdx.x / 16;
  int row0 = blockIdx.y * BM, col0 = blockIdx.x * BN;
  float acc[TM][TN] = {0};

  for (int k0 = 0; k0 < K; k0 += BK) {
    for (int l = threadIdx.x; l < BM * BK; l += 256) {        // A tile: 64 x 16
      int m = l / BK, k = l % BK, gr = row0 + m, gk = k0 + k;
      As[m][k] = (gr < M && gk < K) ? A[gr * K + gk] : 0.f;
    }
    for (int l = threadIdx.x; l < BK * BN; l += 256) {        // B tile: 16 x 64
      int k = l / BN, n = l % BN, gk = k0 + k, gc = col0 + n;
      Bs[k][n] = (gk < K && gc < N) ? B[gk * N + gc] : 0.f;
    }
    __syncthreads();
    for (int k = 0; k < BK; k++) {
      float a[TM], b[TN];
      for (int i = 0; i < TM; i++) a[i] = As[ty * TM + i][k];
      for (int j = 0; j < TN; j++) b[j] = Bs[k][tx * TN + j];
      for (int i = 0; i < TM; i++)
        for (int j = 0; j < TN; j++) acc[i][j] += a[i] * b[j];
    }
    __syncthreads();
  }
  for (int i = 0; i < TM; i++)
    for (int j = 0; j < TN; j++) {
      int r = row0 + ty * TM + i, c = col0 + tx * TN + j;
      if (r < M && c < N) C[r * N + c] = acc[i][j];
    }
}
// launch: sgemm_reg<<<dim3((N + BN - 1) / BN, (M + BM - 1) / BM), 256>>>(A, B, C, M, N, K);
```

## Lab: run all three and compare with cuBLAS

In [ ]:
import torch, triton
a = torch.randn(4096, 4096, device="cuda"); b = torch.randn(4096, 4096, device="cuda")
ms = triton.testing.do_bench(lambda: a @ b)
print(f"cuBLAS FP32: {2 * 4096**3 / (ms / 1e3) / 1e12:.2f} TFLOP/s")

## Why this matters for inference

## What you can now do

## Common mistakes

## Exercise

## Going deeper

---
Code from [Inference Academy](https://inferenceacademy.com), MIT licensed. Lesson: https://inferenceacademy.com/learn/kern-cuda/shared-memory-tiling-and-a-tiled-matmul